# ATE train phase 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
import ast
import json
import random
from pathlib import Path
from collections import Counter

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed,
)

set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 3. Load dataset

In [ ]:
import pandas as pd
import ast
import re

train_path = '/content/drive/MyDrive/absa_self_train_phase2/train.csv'
test_path = '/content/drive/MyDrive/absa_self_train_phase2/gold_test.csv'

sentence_col = "sentence_text"
aspects_col = "aspects"
confidences_col = "confidences"

def load_phase2_data(path):
    # Using low_memory=False to avoid DtypeWarning for mixed types
    df = pd.read_csv(path, low_memory=False)

    def parse_list_string(x):
        if pd.isna(x) or not isinstance(x, str):
            return []
        # Fix NumPy style strings like '[0.1 0.2]' by adding commas: '[0.1, 0.2]'
        # This makes them compatible with ast.literal_eval
        s = x.strip()
        if s.startswith('[') and s.endswith(']'):
            # Replace spaces (one or more) between numbers/elements with a comma
            s = re.sub(r'(\d|\.)\s+(\d|\.|-)', r'\1, \2', s)
        try:
            return ast.literal_eval(s)
        except Exception:
            return []

    for col in [aspects_col, confidences_col]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list_string)
    return df

train_df = load_phase2_data(train_path)
test_df = load_phase2_data(test_path)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
display(train_df.head(2))

Train shape: (1998826, 8)
Test shape: (800, 8)


,parent_asin,sentence_id,sentence_text,rating,category_name,gate_confidence,aspects,confidences
0,B01ARB7MPW,4,this aluminum credit card holder looked like j...,5.0,Office_Products,0.0239391941577196,[],[]
1,B00V96M6GU,1,also the material is of lower quality than exp...,3.0,Office_Products,0.9920926690101624,[material],[0.98056763]


# Part A — Gate

In [ ]:
# Runtime knobs for phase 2.
# This cell keeps the phase-2 dataframe schema intact:
# parent_asin, sentence_id, sentence_text, rating, category_name, gate_confidence, aspects, confidences.

import os
import gc
import inspect
import shutil
from datetime import datetime

FAST_MODE = True
MEMORY_SAFE_MODE = True

# Speed / memory knobs.
USE_GRADIENT_CHECKPOINTING = False  # Turn True only if GPU VRAM OOM; it is slower.
AUTO_FIND_BATCH_SIZE = True
USE_FUSED_ADAMW = torch.cuda.is_available()
USE_TORCH_COMPILE = False

# Sequence length. If aspects are short, 160/128 is the safest speed lever.
GATE_MAX_LENGTH = 192
ATE_MAX_LENGTH = 192

# Conservative defaults for Colab RAM stability. Increase only after RAM stays flat.
GATE_TRAIN_BATCH_SIZE = 64
GATE_EVAL_BATCH_SIZE = 64
ATE_TRAIN_BATCH_SIZE = 64
ATE_EVAL_BATCH_SIZE = 64

GATE_GRAD_ACCUM = 1 if torch.cuda.is_available() else 4
ATE_GRAD_ACCUM = 1 if torch.cuda.is_available() else 8

# Early stopping uses validation loss. Epochs are max epochs, not guaranteed epochs.
GATE_EPOCHS = 3
ATE_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0

# Validation split from train only. Test remains untouched for final reporting.
GATE_VAL_SIZE = 0.05
ATE_VAL_SIZE = 0.05

# Checkpointing.
# The previous Drive sync around ~1000 steps can stall training because full Trainer
# checkpoints include optimizer/scheduler state and are large. Local save is still used;
# Drive sync is throttled to avoid RAM/I/O spikes.
CHECKPOINT_STRATEGY = "steps"  # "steps" gives mid-epoch resume; "epoch" is lighter.
CHECKPOINT_STEPS = 2000         # Increase if Drive I/O is still too slow; lower if resume granularity matters.
SAVE_TOTAL_LIMIT = 2
DRIVE_CHECKPOINT_KEEP_LAST = 1
SYNC_CHECKPOINTS_TO_DRIVE = True
DRIVE_SYNC_EVERY_N_CHECKPOINTS = 1  # With CHECKPOINT_STEPS=2000, sync every 2000 steps.
RESUME_FROM_DRIVE_CHECKPOINT = False  # Set True after a Colab disconnect to resume from latest Drive checkpoint.

# RAM-safe dataloader settings.
# num_workers>0 can duplicate the Dataset object in worker processes. With large text/token data,
# that is a common reason system RAM climbs toward max during long training.
DATALOADER_NUM_WORKERS = 0
DATALOADER_PIN_MEMORY = torch.cuda.is_available()
GROUP_BY_LENGTH = False

LOGGING_STEPS = 200 if FAST_MODE else 100

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

gate_model_name = "roberta-base"

gate_output_dir = "/content/model_cache_phase2/ate_gate_teacher_phase2/trainer"
gate_checkpoint_drive_dir = "/content/drive/MyDrive/ate_gate_teacher_phase2/checkpoints"
gate_best_dir = "/content/drive/MyDrive/ate_gate_teacher_phase2/best_eval_loss"
gate_final_dir = "/content/drive/MyDrive/ate_gate_teacher_phase2/final"

phase2_columns = [
    "parent_asin",
    "sentence_id",
    "sentence_text",
    "rating",
    "category_name",
    "gate_confidence",
    "aspects",
    "confidences",
]

for df_name, df in [("train_df", train_df), ("test_df", test_df)]:
    missing_cols = [col for col in phase2_columns if col not in df.columns]
    extra_cols = [col for col in df.columns if col not in phase2_columns]

    if missing_cols:
        raise ValueError(f"{df_name} is missing required phase-2 columns: {missing_cols}")

    if extra_cols:
        print(f"Warning: {df_name} has extra columns not used by this notebook: {extra_cols}")

def ensure_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []

        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
            if isinstance(parsed, tuple):
                return list(parsed)
            return [parsed]
        except Exception:
            return [value]

    return [value]

def get_aspects_from_value(value):
    raw_aspects = ensure_list(value)
    return [str(x).strip() for x in raw_aspects if str(x).strip()]

def get_confidences_from_value(value):
    raw_confidences = ensure_list(value)
    values = []

    for item in raw_confidences:
        try:
            values.append(float(item))
        except Exception:
            values.append(np.nan)

    return values

def get_aspects(row):
    return get_aspects_from_value(row[aspects_col])

def get_confidences(row):
    return get_confidences_from_value(row[confidences_col])

def aspect_presence_labels(dataframe):
    # Local labels only; this does not recreate old phase-1 columns.
    return np.array(
        [int(len(get_aspects_from_value(value)) > 0) for value in dataframe[aspects_col].tolist()],
        dtype=np.int64,
    )

def make_train_val_indices(labels, val_size=0.1, seed=42):
    labels = np.asarray(labels)
    indices = np.arange(len(labels))

    if len(indices) < 2:
        return indices, indices[:0]

    counts = pd.Series(labels).value_counts()
    can_stratify = len(counts) > 1 and counts.min() >= 2
    stratify = labels if can_stratify else None

    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_size,
        random_state=seed,
        stratify=stratify,
    )

    return train_idx, val_idx

def trainer_tokenizer_kwargs(tokenizer):
    # Transformers renamed Trainer(tokenizer=...) to Trainer(processing_class=...).
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        return {"processing_class": tokenizer}

    return {"tokenizer": tokenizer}

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

gate_tokenizer = AutoTokenizer.from_pretrained(gate_model_name, use_fast=True)

class GateTextDataset(Dataset):
    """RAM-safe dataset: keep raw text + labels only; tokenize per batch in collator."""

    def __init__(self, dataframe, labels=None):
        self.texts = dataframe[sentence_col].fillna("").astype(str).tolist()
        self.labels = np.asarray(labels if labels is not None else aspect_presence_labels(dataframe), dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "labels": int(self.labels[idx]),
        }

class GateBatchCollator:
    def __init__(self, tokenizer, max_length=192):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        texts = [x["text"] for x in features]
        labels = torch.tensor([x["labels"] for x in features], dtype=torch.long)
        enc = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt",
        )
        enc["labels"] = labels
        return enc

gate_all_train_labels = aspect_presence_labels(train_df)
gate_test_labels = aspect_presence_labels(test_df)
gate_train_idx, gate_val_idx = make_train_val_indices(gate_all_train_labels, val_size=GATE_VAL_SIZE)

gate_train_dataset = GateTextDataset(
    train_df.iloc[gate_train_idx].reset_index(drop=True),
    labels=gate_all_train_labels[gate_train_idx],
)
gate_val_dataset = GateTextDataset(
    train_df.iloc[gate_val_idx].reset_index(drop=True),
    labels=gate_all_train_labels[gate_val_idx],
)
gate_test_dataset = GateTextDataset(
    test_df,
    labels=gate_test_labels,
)

gate_data_collator = GateBatchCollator(tokenizer=gate_tokenizer, max_length=GATE_MAX_LENGTH)

print("Gate train samples:", len(gate_train_dataset))
print("Gate val samples  :", len(gate_val_dataset))
print("Gate test samples :", len(gate_test_dataset))
print("Gate full-train label counts:", pd.Series(gate_all_train_labels).value_counts().sort_index().to_dict())
print("Gate val label counts       :", pd.Series(gate_all_train_labels[gate_val_idx]).value_counts().sort_index().to_dict())
print("Gate test label counts      :", pd.Series(gate_test_labels).value_counts().sort_index().to_dict())
print("Fast mode:", FAST_MODE, "| memory safe:", MEMORY_SAFE_MODE)
print("Gate batch/accum/max_epochs:", GATE_TRAIN_BATCH_SIZE, GATE_GRAD_ACCUM, GATE_EPOCHS)
print("ATE batch/accum/max_epochs :", ATE_TRAIN_BATCH_SIZE, ATE_GRAD_ACCUM, ATE_EPOCHS)
print("Checkpoint strategy:", CHECKPOINT_STRATEGY, "| steps:", CHECKPOINT_STEPS, "| sync to Drive:", SYNC_CHECKPOINTS_TO_DRIVE)
print("Dataloader workers:", DATALOADER_NUM_WORKERS, "| group_by_length:", GROUP_BY_LENGTH)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Gate train samples: 1898884
Gate val samples  : 99942
Gate test samples : 800
Gate full-train label counts: {0: 851318, 1: 1147508}
Gate val label counts       : {0: 42566, 1: 57376}
Gate test label counts      : {0: 346, 1: 454}
Fast mode: True | memory safe: True
Gate batch/accum/max_epochs: 64 1 3
ATE batch/accum/max_epochs : 64 1 3
Checkpoint strategy: steps | steps: 2000 | sync to Drive: True
Dataloader workers: 0 | group_by_length: False


In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
import torch
import os
import json
import shutil
import gc


def build_training_args(
    output_dir,
    lr=2e-5,
    epochs=3,
    train_bs=4,
    eval_bs=8,
    grad_accum=1,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    prediction_loss_only=True,
):
    strategy = CHECKPOINT_STRATEGY
    eval_steps = eval_steps or CHECKPOINT_STEPS
    save_steps = save_steps or eval_steps

    kwargs = dict(
        output_dir=output_dir,
        seed=42,

        do_eval=True,
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,

        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=grad_accum,

        fp16=torch.cuda.is_available(),
        tf32=False, # Changed from torch.cuda.is_available() to False

        logging_strategy="steps",
        logging_steps=logging_steps or LOGGING_STEPS,

        save_strategy=strategy,
        evaluation_strategy=strategy,
        eval_strategy=strategy,
        save_total_limit=SAVE_TOTAL_LIMIT,
        load_best_model_at_end=True,
        metric_for_best_model=metric_for_best_model,
        greater_is_better=greater_is_better,

        # Important for RAM stability:
        # - prediction_loss_only avoids storing logits during validation; early stopping only needs eval_loss.
        # - num_workers=0 avoids duplicating large dataset objects into worker processes.
        prediction_loss_only=prediction_loss_only,
        report_to="none",
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=DATALOADER_PIN_MEMORY,
        remove_unused_columns=False,
        group_by_length=GROUP_BY_LENGTH,
        auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
        save_safetensors=True,
        eval_accumulation_steps=8,
    )

    if strategy == "steps":
        kwargs["eval_steps"] = eval_steps
        kwargs["save_steps"] = save_steps

    if USE_FUSED_ADAMW:
        kwargs["optim"] = "adamw_torch_fused"

    if USE_TORCH_COMPILE:
        kwargs["torch_compile"] = True

    # Keep compatibility with older/newer Colab Transformers versions.
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in sig:
        kwargs.pop("eval_strategy", None)
    if "evaluation_strategy" not in sig:
        kwargs.pop("evaluation_strategy", None)

    filtered = {k: v for k, v in kwargs.items() if k in sig}
    return TrainingArguments(**filtered)


def copy_model_dir_to_drive(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        raise FileNotFoundError(f"Local model directory not found: {src_dir}")

    os.makedirs(os.path.dirname(dst_dir), exist_ok=True)

    tmp_dir = dst_dir.rstrip("/") + ".tmp"
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.copytree(src_dir, tmp_dir)

    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir, ignore_errors=True)
    os.rename(tmp_dir, dst_dir)


class MemoryCleanupCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, **kwargs):
        cleanup_memory()
        return control

    def on_save(self, args, state, control, **kwargs):
        cleanup_memory()
        return control


class DriveCheckpointSyncCallback(TrainerCallback):
    """Copy throttled local Trainer checkpoints to Drive for resume after Colab disconnect.

    Full Trainer checkpoints are large because they include optimizer/scheduler state.
    Throttling avoids the slowdown/RAM pressure that can happen when syncing every save.
    """

    def __init__(self, drive_dir, tokenizer=None, keep_last=1, enabled=True, sync_every_n_checkpoints=1):
        self.drive_dir = drive_dir
        self.tokenizer = tokenizer
        self.keep_last = keep_last
        self.enabled = enabled
        self.sync_every_n_checkpoints = max(int(sync_every_n_checkpoints), 1)
        self._save_count = 0

    @staticmethod
    def _checkpoint_step(path):
        name = os.path.basename(path.rstrip("/"))
        match = re.search(r"checkpoint-(\d+)$", name)
        return int(match.group(1)) if match else -1

    def _prune_old_drive_checkpoints(self):
        if self.keep_last is None or self.keep_last <= 0 or not os.path.exists(self.drive_dir):
            return

        checkpoint_dirs = [
            os.path.join(self.drive_dir, name)
            for name in os.listdir(self.drive_dir)
            if name.startswith("checkpoint-") and os.path.isdir(os.path.join(self.drive_dir, name))
        ]
        checkpoint_dirs = sorted(checkpoint_dirs, key=self._checkpoint_step)

        for old_dir in checkpoint_dirs[:-self.keep_last]:
            shutil.rmtree(old_dir, ignore_errors=True)

    def on_save(self, args, state, control, **kwargs):
        cleanup_memory()

        if not self.enabled or not state.is_world_process_zero:
            return control

        self._save_count += 1
        if self._save_count % self.sync_every_n_checkpoints != 0:
            print(f"\nSkipped Drive checkpoint sync at step {state.global_step}; local checkpoint still exists.")
            return control

        local_checkpoint = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.exists(local_checkpoint):
            return control

        os.makedirs(self.drive_dir, exist_ok=True)
        drive_checkpoint = os.path.join(self.drive_dir, f"checkpoint-{state.global_step}")
        tmp_checkpoint = drive_checkpoint + ".tmp"

        if os.path.exists(tmp_checkpoint):
            shutil.rmtree(tmp_checkpoint, ignore_errors=True)

        shutil.copytree(local_checkpoint, tmp_checkpoint)

        if self.tokenizer is not None:
            self.tokenizer.save_pretrained(tmp_checkpoint)

        if os.path.exists(drive_checkpoint):
            shutil.rmtree(drive_checkpoint, ignore_errors=True)
        os.rename(tmp_checkpoint, drive_checkpoint)

        with open(os.path.join(self.drive_dir, "latest_checkpoint.json"), "w") as f:
            json.dump(
                {
                    "latest_checkpoint": drive_checkpoint,
                    "global_step": int(state.global_step),
                    "epoch": float(state.epoch) if state.epoch is not None else None,
                    "synced_at": datetime.now().isoformat(timespec="seconds"),
                    "note": "Full Trainer checkpoint for resume. Local checkpoints may be newer if sync was throttled.",
                },
                f,
                indent=2,
            )

        self._prune_old_drive_checkpoints()
        cleanup_memory()
        print(f"\nSynced checkpoint to Drive: {drive_checkpoint}")

        return control


def find_latest_drive_checkpoint(drive_dir):
    if not os.path.exists(drive_dir):
        return None

    latest_json = os.path.join(drive_dir, "latest_checkpoint.json")
    if os.path.exists(latest_json):
        try:
            with open(latest_json, "r") as f:
                info = json.load(f)
            path = info.get("latest_checkpoint")
            if path and os.path.exists(path):
                return path
        except Exception:
            pass

    checkpoint_dirs = [
        os.path.join(drive_dir, name)
        for name in os.listdir(drive_dir)
        if name.startswith("checkpoint-") and os.path.isdir(os.path.join(drive_dir, name))
    ]

    if not checkpoint_dirs:
        return None

    return sorted(checkpoint_dirs, key=DriveCheckpointSyncCallback._checkpoint_step)[-1]


def prepare_resume_checkpoint(drive_dir, local_output_dir, enabled=False):
    if not enabled:
        return None

    drive_checkpoint = find_latest_drive_checkpoint(drive_dir)
    if drive_checkpoint is None:
        print(f"No Drive checkpoint found under: {drive_dir}")
        return None

    os.makedirs(local_output_dir, exist_ok=True)
    local_checkpoint = os.path.join(local_output_dir, os.path.basename(drive_checkpoint))

    if os.path.exists(local_checkpoint):
        shutil.rmtree(local_checkpoint, ignore_errors=True)

    shutil.copytree(drive_checkpoint, local_checkpoint)
    print(f"Prepared local resume checkpoint from Drive: {local_checkpoint}")

    return local_checkpoint


def compute_gate_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )
    acc = accuracy_score(labels, preds)

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    if len(np.unique(labels)) == 2:
        try:
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
            metrics["roc_auc"] = roc_auc_score(labels, probs)
        except Exception:
            pass

    return metrics


gate_model = AutoModelForSequenceClassification.from_pretrained(
    gate_model_name,
    num_labels=2,
    id2label={0: "no_aspect", 1: "with_aspect"},
    label2id={"no_aspect": 0, "with_aspect": 1},
).to(device)

if USE_GRADIENT_CHECKPOINTING and hasattr(gate_model, "gradient_checkpointing_enable"):
    gate_model.gradient_checkpointing_enable()

gate_args = build_training_args(
    output_dir=gate_output_dir,
    lr=1e-5,
    epochs=GATE_EPOCHS,
    train_bs=GATE_TRAIN_BATCH_SIZE,
    eval_bs=GATE_EVAL_BATCH_SIZE,
    grad_accum=GATE_GRAD_ACCUM,
    logging_steps=LOGGING_STEPS,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

gate_trainer = Trainer(
    model=gate_model,
    args=gate_args,
    train_dataset=gate_train_dataset,
    eval_dataset=gate_val_dataset,
    data_collator=gate_data_collator,
    # No compute_metrics during training: early stopping only needs eval_loss.
    # This avoids accumulating validation logits in RAM.
    compute_metrics=None,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ),
        MemoryCleanupCallback(),
        DriveCheckpointSyncCallback(
            drive_dir=gate_checkpoint_drive_dir,
            tokenizer=gate_tokenizer,
            keep_last=DRIVE_CHECKPOINT_KEEP_LAST,
            enabled=SYNC_CHECKPOINTS_TO_DRIVE,
            sync_every_n_checkpoints=DRIVE_SYNC_EVERY_N_CHECKPOINTS,
        ),
    ],
    **trainer_tokenizer_kwargs(gate_tokenizer),
)

gate_resume_checkpoint = prepare_resume_checkpoint(
    gate_checkpoint_drive_dir,
    gate_output_dir,
    enabled=RESUME_FROM_DRIVE_CHECKPOINT,
)

gate_train_result = gate_trainer.train(resume_from_checkpoint=gate_resume_checkpoint)

# Because load_best_model_at_end=True, this saves the best eval-loss model, not merely the final step.
gate_trainer.save_model(gate_best_dir)
gate_tokenizer.save_pretrained(gate_best_dir)

gate_trainer.save_model(gate_final_dir)
gate_tokenizer.save_pretrained(gate_final_dir)

with open(os.path.join(gate_best_dir, "training_summary.json"), "w") as f:
    json.dump(
        {
            "best_metric": float(gate_trainer.state.best_metric) if gate_trainer.state.best_metric is not None else None,
            "best_model_checkpoint": gate_trainer.state.best_model_checkpoint,
            "global_step": int(gate_trainer.state.global_step),
            "max_epochs": GATE_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "checkpoint_steps": CHECKPOINT_STEPS,
            "drive_sync_every_n_checkpoints": DRIVE_SYNC_EVERY_N_CHECKPOINTS,
            "train_result": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in gate_train_result.metrics.items()},
        },
        f,
        indent=2,
    )

cleanup_memory()
print("Best gate eval_loss:", gate_trainer.state.best_metric)
print("Best local checkpoint:", gate_trainer.state.best_model_checkpoint)
print("Saved best gate model to Drive:", gate_best_dir)
print("Saved final-loaded gate model to Drive:", gate_final_dir)
print("Drive checkpoints:", gate_checkpoint_drive_dir)


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 8. Evaluate gate teacher on test

In [ ]:
from sklearn.metrics import f1_score

gate_best_dir = "/content/drive/MyDrive/ate_gate_teacher_phase2/best_eval_loss"

gate_tokenizer = AutoTokenizer.from_pretrained(gate_best_dir, use_fast=True)
gate_model = AutoModelForSequenceClassification.from_pretrained(gate_best_dir).to(device)
gate_model.eval()


def predict_gate_proba(sentences, batch_size=64 if torch.cuda.is_available() else 16):
    probs_all = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="Gate predicting"):
        batch = [str(x) for x in sentences[i:i + batch_size]]

        enc = gate_tokenizer(
            batch,
            truncation=True,
            max_length=GATE_MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            logits = gate_model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

        probs_all.extend(probs.tolist())

    return np.array(probs_all)


X_gate_test = test_df[sentence_col].astype(str).tolist()
y_gate_test = gate_test_labels

gate_test_proba = predict_gate_proba(X_gate_test)

default_threshold = 0.5
gate_test_pred = (gate_test_proba >= default_threshold).astype(int)

print("Evaluation target: len(aspects) > 0")
print(f"Threshold: {default_threshold}")
print("Accuracy:", round(accuracy_score(y_gate_test, gate_test_pred), 4))
print("F1 binary / with_aspect:", round(f1_score(y_gate_test, gate_test_pred, pos_label=1, zero_division=0), 4))
print("F1 macro:", round(f1_score(y_gate_test, gate_test_pred, average="macro", zero_division=0), 4))
print("F1 weighted:", round(f1_score(y_gate_test, gate_test_pred, average="weighted", zero_division=0), 4))
print()

print(classification_report(
    y_gate_test,
    gate_test_pred,
    target_names=["no_aspect", "with_aspect"],
    digits=4,
    zero_division=0,
))

try:
    print("ROC-AUC:", round(roc_auc_score(y_gate_test, gate_test_proba), 4))
except Exception as e:
    print("ROC-AUC unavailable:", e)


threshold_rows = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    pred = (gate_test_proba >= threshold).astype(int)

    with_aspect_precision, with_aspect_recall, with_aspect_f1, _ = precision_recall_fscore_support(
        y_gate_test,
        pred,
        labels=[1],
        average="binary",
        zero_division=0,
    )

    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_gate_test, pred),
        "with_aspect_precision": with_aspect_precision,
        "with_aspect_recall": with_aspect_recall,
        "with_aspect_f1": with_aspect_f1,
        "macro_f1": f1_score(y_gate_test, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_gate_test, pred, average="weighted", zero_division=0),
        "pass_rate": pred.mean(),
    })

gate_threshold_df = pd.DataFrame(threshold_rows)

display(
    gate_threshold_df.style.format({
        "threshold": "{:.2f}",
        "accuracy": "{:.4f}",
        "with_aspect_precision": "{:.4f}",
        "with_aspect_recall": "{:.4f}",
        "with_aspect_f1": "{:.4f}",
        "macro_f1": "{:.4f}",
        "weighted_f1": "{:.4f}",
        "pass_rate": "{:.4f}",
    })
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Gate predicting:   0%|          | 0/13 [00:00<?, ?it/s]

Evaluation target: len(aspects) > 0
Threshold: 0.5
Accuracy: 0.835
F1 binary / with_aspect: 0.8622
F1 macro: 0.8283
F1 weighted: 0.8329

              precision    recall  f1-score   support

   no_aspect     0.8615    0.7370    0.7944       346
 with_aspect     0.8194    0.9097    0.8622       454

    accuracy                         0.8350       800
   macro avg     0.8405    0.8233    0.8283       800
weighted avg     0.8376    0.8350    0.8329       800

ROC-AUC: 0.9156


,threshold,accuracy,with_aspect_precision,with_aspect_recall,with_aspect_f1,macro_f1,weighted_f1,pass_rate
0,0.10,0.8213,0.7951,0.9229,0.8542,0.8116,0.8174,0.6587
1,0.20,0.8287,0.8066,0.9185,0.8589,0.8206,0.8257,0.6462
2,0.30,0.8337,0.8141,0.9163,0.8622,0.8264,0.8312,0.6388
3,0.40,0.8337,0.8166,0.9119,0.8616,0.8267,0.8314,0.6338
4,0.50,0.8350,0.8194,0.9097,0.8622,0.8283,0.8329,0.6300
5,0.60,0.8325,0.8226,0.8987,0.8589,0.8264,0.8308,0.6200
6,0.70,0.8350,0.8313,0.8899,0.8596,0.8298,0.8338,0.6075
7,0.80,0.8363,0.8386,0.8811,0.8593,0.8317,0.8355,0.5962
8,0.90,0.8363,0.8458,0.8700,0.8578,0.8324,0.8358,0.5837
9,0.95,0.8350,0.8546,0.8546,0.8546,0.8319,0.8350,0.5675


# Part B — ATE

In [ ]:
TOKEN_RE = re.compile(r"\b\w+(?:'\w+)?\b", flags=re.UNICODE)

label_list = ["O", "B-ASP", "I-ASP"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_words(text):
    return [m.group(0) for m in TOKEN_RE.finditer(clean_text(text))]

def normalize_aspect(text):
    text = clean_text(text).lower()
    text = re.sub(r"_+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def aspect_to_tokens(aspect):
    normalized = normalize_aspect(aspect)
    if not normalized:
        return []
    return tokenize_words(normalized)

GOLD_POS_WEIGHT = 1.0
PSEUDO_POS_WEIGHT_MIN = 0.6
PSEUDO_POS_WEIGHT_MAX = 0.9

GOLD_O_WEIGHT = 1.0
PSEUDO_O_WEIGHT = 0.15

PSEUDO_CONF_FLOOR = 0.85
PSEUDO_CONF_CEIL = 1.0


def is_gold_confidences(confidences):
    confidences = [c for c in confidences if not pd.isna(c)]
    return len(confidences) > 0 and all(float(c) >= 0.999 for c in confidences)


def confidence_to_positive_weight(conf, is_gold_row=False):
    if is_gold_row:
        return GOLD_POS_WEIGHT

    if pd.isna(conf):
        conf = PSEUDO_CONF_FLOOR

    conf = float(conf)
    conf = min(max(conf, PSEUDO_CONF_FLOOR), PSEUDO_CONF_CEIL)

    ratio = (conf - PSEUDO_CONF_FLOOR) / max(PSEUDO_CONF_CEIL - PSEUDO_CONF_FLOOR, 1e-8)

    return PSEUDO_POS_WEIGHT_MIN + ratio * (
        PSEUDO_POS_WEIGHT_MAX - PSEUDO_POS_WEIGHT_MIN
    )


def create_bio_labels(sentence, aspects, confidences=None):
    tokens = tokenize_words(sentence)
    labels = ["O"] * len(tokens)

    confidences = confidences or []
    is_gold_row = is_gold_confidences(confidences)

    o_weight = GOLD_O_WEIGHT if is_gold_row else PSEUDO_O_WEIGHT
    weights = [o_weight] * len(tokens)

    token_norm = [normalize_aspect(tok) for tok in tokens]
    unmatched = []

    for aspect_idx, aspect in enumerate(aspects):
        asp_tokens = aspect_to_tokens(aspect)

        if not asp_tokens:
            continue

        if aspect_idx < len(confidences):
            conf = confidences[aspect_idx]
        else:
            conf = np.nan

        pos_weight = confidence_to_positive_weight(conf, is_gold_row=is_gold_row)

        n = len(asp_tokens)
        found = False

        for i in range(0, len(tokens) - n + 1):
            if token_norm[i:i + n] == asp_tokens:
                labels[i] = "B-ASP"
                weights[i] = pos_weight

                for j in range(i + 1, i + n):
                    labels[j] = "I-ASP"
                    weights[j] = pos_weight

                found = True

        if not found:
            unmatched.append(aspect)

    return tokens, labels, weights, unmatched

def add_ate_columns(dataframe, desc):
    rows = []
    n_aspects = 0
    n_unmatched = 0
    unmatched_counter = Counter()

    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=desc):
        aspects = get_aspects(row)
        confidences = get_confidences(row)
        tokens, labels, weights, unmatched = create_bio_labels(
            row[sentence_col],
            aspects,
            confidences,
        )

        n_aspects += len(aspects)
        n_unmatched += len(unmatched)

        for aspect in unmatched:
            unmatched_counter[aspect] += 1

        rows.append({
            "source_index": idx,
            sentence_col: row[sentence_col],
            "tokens": tokens,
            "word_labels": labels,
            aspects_col: aspects,
            confidences_col: confidences,
            "unmatched_aspects": unmatched,
            "word_weights": weights,
        })

    out = pd.DataFrame(rows)
    match_rate = 1 - n_unmatched / max(n_aspects, 1)

    return out, {
        "n_aspects": n_aspects,
        "n_unmatched": n_unmatched,
        "match_rate": match_rate,
        "unmatched_counter": unmatched_counter,
    }

ate_train_source = train_df.loc[gate_all_train_labels.astype(bool)].copy().reset_index(drop=True)
ate_test_source = test_df.loc[gate_test_labels.astype(bool)].copy().reset_index(drop=True)

ate_train_df, ate_train_info = add_ate_columns(ate_train_source, "BIO train")
ate_test_df, ate_test_info = add_ate_columns(ate_test_source, "BIO test")

print("ATE train:", ate_train_df.shape, {k: v for k, v in ate_train_info.items() if k != "unmatched_counter"})
print("ATE test :", ate_test_df.shape, {k: v for k, v in ate_test_info.items() if k != "unmatched_counter"})
print("Top train unmatched:", ate_train_info["unmatched_counter"].most_common(20))
print("Top test unmatched :", ate_test_info["unmatched_counter"].most_common(20))


BIO train:   0%|          | 0/1147508 [00:00<?, ?it/s]

BIO test:   0%|          | 0/454 [00:00<?, ?it/s]

ATE train: (1147508, 8) {'n_aspects': 1206971, 'n_unmatched': 493174, 'match_rate': 0.5913953193573003}
ATE test : (454, 8) {'n_aspects': 656, 'n_unmatched': 24, 'match_rate': 0.9634146341463414}
Top train unmatched: [('0.98705274', 975), ('qualityprice', 908), ('pricequality', 686), ('gamegames', 615), ('gamegraphics', 552), ('gameapp', 521), ('0.98953152', 365), ('valueprice', 334), ('soundbass', 324), ('appgame', 324), ('gamelevels', 322), ('soundprice', 280), ('soundquality', 235), ('gameupdate', 212), ('0.98802841', 210), ('soundbattery life', 207), ('graphicsgame', 206), ('soundspeaker', 191), ('sound qualitybattery life', 188), ('gameads', 183)]
Top test unmatched : [('game', 2), ('it', 2), ('installation', 2), ('nba game app', 1), ('volume', 1), ('usb hub', 1), ('fit', 1), ('value', 1), ('situation', 1), ('delivery', 1), ('advertisement', 1), ('conducción', 1), ('conference calling', 1), ('compatibility with apple and windows', 1), ('assembly', 1), ('packaging', 1), ('labeling'

In [ ]:

for name, part in [("train", ate_train_df), ("test", ate_test_df)]:
    c = Counter()
    for labels in part["word_labels"]:
        c.update(labels)
    print(name, c)

    if name == "train" and c["B-ASP"] == 0:
        raise ValueError("No B-ASP labels in train. Check aspect matching against the aspects column.")

display(
    ate_train_df[ate_train_df["word_labels"].apply(lambda xs: "B-ASP" in xs)][
        [sentence_col, "tokens", "word_labels", aspects_col, confidences_col]
    ].head(10)
)


train Counter({'O': 17264567, 'B-ASP': 797966, 'I-ASP': 222063})
test Counter({'O': 6386, 'B-ASP': 678, 'I-ASP': 253})


,sentence_text,tokens,word_labels,aspects,confidences
0,also the material is of lower quality than exp...,"[also, the, material, is, of, lower, quality, ...","[O, O, B-ASP, O, O, O, O, O, O]",[material],[0.98056763]
1,the ink or tips dried up quickly i didnt,"[the, ink, or, tips, dried, up, quickly, i, di...","[O, B-ASP, I-ASP, I-ASP, O, O, O, O, O]",[ink or tips],[0.94926622]
2,the flamingo design just adds a [GENERIC_NOUN]...,"[the, flamingo, design, just, adds, a, GENERIC...","[O, B-ASP, I-ASP, O, O, O, O, O, O, O, O, O]",[flamingo design],[0.94343317]
4,excellent these stickers are think and stick r...,"[excellent, these, stickers, are, think, and, ...","[O, O, B-ASP, O, O, O, O, O, O]",[stickers],[0.98821718]
7,great [GENERIC_NOUN] ive ordered ink from seve...,"[great, GENERIC_NOUN, ive, ordered, ink, from,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[cartridges],[0.9749741]
8,the photos do not adhere to the plastic do not...,"[the, photos, do, not, adhere, to, the, plasti...","[O, B-ASP, O, O, O, O, O, O, O, O, O, O, O, O, O]",[photos],[0.96958959]
10,these tabbed az inserts were perfect for what ...,"[these, tabbed, az, inserts, were, perfect, fo...","[O, B-ASP, I-ASP, I-ASP, O, O, O, O, O, O]",[tabbed az inserts],[0.95246156]
11,54th can come off really unwieldy on the wrong...,"[54th, can, come, off, really, unwieldy, on, t...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[color],[0.91225314]
13,[GENERIC_NOUN] functions as it should not a ba...,"[GENERIC_NOUN, functions, as, it, should, not,...","[O, O, O, O, O, O, O, O, B-ASP]",[printer],[0.90365726]
14,the software installation is slow [GENERIC_NOU...,"[the, software, installation, is, slow, GENERI...","[O, B-ASP, I-ASP, O, O, O, O, O, O, O, O, O, O...",[software installation],[0.94182196]


In [ ]:
ate_model_name = "roberta-base"

ate_tokenizer = AutoTokenizer.from_pretrained(ate_model_name, use_fast=True)

class ATERawDataset(Dataset):
    """RAM-safe ATE dataset: store raw tokens + word labels + word weights."""

    def __init__(self, dataframe):
        self.token_rows = dataframe["tokens"].tolist()
        self.label_rows = dataframe["word_labels"].tolist()
        self.weight_rows = dataframe["word_weights"].tolist()

    def __len__(self):
        return len(self.token_rows)

    def __getitem__(self, idx):
        return {
            "tokens": self.token_rows[idx],
            "word_labels": self.label_rows[idx],
            "word_weights": self.weight_rows[idx],
        }

class ATEBatchCollator:
    def __init__(self, tokenizer, max_length=192):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        token_rows = [x["tokens"] for x in features]
        label_rows = [x["word_labels"] for x in features]
        weight_rows = [x["word_weights"] for x in features]

        enc = self.tokenizer(
            token_rows,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt",
        )

        aligned_label_batch = []
        aligned_weight_batch = []
        for batch_idx, word_labels in enumerate(label_rows):
            word_weights = weight_rows[batch_idx]

            word_label_ids = [label2id[x] for x in word_labels]
            word_ids = enc.word_ids(batch_index=batch_idx)

            aligned_labels = []
            aligned_weights = []
            previous_word_id = None

            for word_id in word_ids:
                if word_id is None:
                    aligned_labels.append(-100)
                    aligned_weights.append(0.0)
                elif word_id != previous_word_id:
                    if word_id < len(word_label_ids):
                        aligned_labels.append(word_label_ids[word_id])
                        aligned_weights.append(float(word_weights[word_id]))
                    else:
                        aligned_labels.append(-100)
                        aligned_weights.append(0.0)
                else:
                    aligned_labels.append(-100)
                    aligned_weights.append(0.0)

                previous_word_id = word_id

            aligned_label_batch.append(aligned_labels)
            aligned_weight_batch.append(aligned_weights)

        enc["labels"] = torch.tensor(aligned_label_batch, dtype=torch.long)
        enc["token_weights"] = torch.tensor(aligned_weight_batch, dtype=torch.float)

        return enc

# Validation split for early stopping. Stratify by whether a row contains any B-ASP tag.
ate_row_labels = np.array(
    [int("B-ASP" in labels) for labels in ate_train_df["word_labels"].tolist()],
    dtype=np.int64,
)
# ate_train_idx, ate_val_idx = make_train_val_indices(ate_row_labels, val_size=ATE_VAL_SIZE)

# ate_train_split_df = ate_train_df.iloc[ate_train_idx].reset_index(drop=True)
# ate_val_df = ate_train_df.iloc[ate_val_idx].reset_index(drop=True)

ate_train_dataset = ATERawDataset(ate_train_df)
ate_val_dataset = ATERawDataset(ate_test_df)
ate_test_dataset = ATERawDataset(ate_test_df)

ate_data_collator = ATEBatchCollator(tokenizer=ate_tokenizer, max_length=ATE_MAX_LENGTH)

print("ATE train samples:", len(ate_train_dataset))
print("ATE val samples  :", len(ate_val_dataset))
print("ATE test samples :", len(ate_test_dataset))
print("ATE val row-label counts:", pd.Series(ate_row_labels[ate_val_idx]).value_counts().sort_index().to_dict())


ATE train samples: 1147508
ATE val samples  : 454
ATE test samples : 454
ATE val row-label counts: {0: 21721, 1: 35655}


## 12. Weighted loss Trainer

In [ ]:
def compute_ate_token_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)
    mask = labels != -100

    y_true = labels[mask]
    y_pred = preds[mask]

    if len(y_true) == 0:
        return {
            "token_accuracy": 0.0,
            "asp_precision": 0.0,
            "asp_recall": 0.0,
            "asp_f1": 0.0,
        }

    token_accuracy = accuracy_score(y_true, y_pred)

    # Report ASP-only token metrics. "O" can dominate token accuracy, so this is more informative.
    asp_labels = [label2id["B-ASP"], label2id["I-ASP"]]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=asp_labels,
        average="micro",
        zero_division=0,
    )

    return {
        "token_accuracy": token_accuracy,
        "asp_precision": precision,
        "asp_recall": recall,
        "asp_f1": f1,
    }


In [ ]:
label_counts = Counter()
for labels in ate_train_df["word_labels"]:
    label_counts.update(labels)

counts = torch.tensor([label_counts[label] for label in label_list], dtype=torch.float)
freq = counts / counts.sum()

class_weights = 1.0 / torch.sqrt(freq)
class_weights = class_weights / class_weights.mean()
class_weights = torch.clamp(class_weights, min=0.25, max=5.0)

print("ATE label counts:", label_counts)
print("Class weights:")
for label, weight in zip(label_list, class_weights.tolist()):
    print(label, round(weight, 4))

class WeightedTokenTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        token_weights = inputs.pop("token_weights", None)

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
            reduction="none",
        )

        flat_logits = logits.view(-1, model.config.num_labels)
        flat_labels = labels.view(-1)

        per_token_loss = loss_fct(flat_logits, flat_labels)

        valid_mask = flat_labels != -100

        if token_weights is not None:
            flat_token_weights = token_weights.view(-1).to(logits.device)
            weighted_loss = per_token_loss[valid_mask] * flat_token_weights[valid_mask]
            denom = flat_token_weights[valid_mask].sum().clamp_min(1e-8)
            loss = weighted_loss.sum() / denom
        else:
            loss = per_token_loss[valid_mask].mean()

        return (loss, outputs) if return_outputs else loss

ATE label counts: Counter({'O': 17264567, 'B-ASP': 797966, 'I-ASP': 222063})
Class weights:
O 0.25
B-ASP 0.9644
I-ASP 1.8282


In [ ]:
def bio_spans(label_names):
    spans = []
    i = 0

    while i < len(label_names):
        if label_names[i] == "B-ASP":
            start = i
            i += 1

            while i < len(label_names) and label_names[i] == "I-ASP":
                i += 1

            end = i - 1
            spans.append((start, end))
        else:
            i += 1

    return set(spans)


def compute_ate_span_metrics(eval_pred):
    predictions, labels = eval_pred

    # Nếu không dùng preprocess_logits_for_metrics thì predictions là logits 3D.
    # Nếu có dùng thì predictions đã là pred_ids 2D.
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        pred_ids = np.argmax(predictions, axis=-1)
    else:
        pred_ids = predictions

    tp = 0
    fp = 0
    fn = 0

    for pred_seq, gold_seq in zip(pred_ids, labels):
        pred_labels = []
        gold_labels = []

        for pred_id, gold_id in zip(pred_seq, gold_seq):
            if gold_id == -100:
                continue

            pred_labels.append(id2label[int(pred_id)])
            gold_labels.append(id2label[int(gold_id)])

        pred_spans = bio_spans(pred_labels)
        gold_spans = bio_spans(gold_labels)

        tp += len(pred_spans & gold_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": f1,
    }


def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]

    return torch.argmax(logits, dim=-1)

In [ ]:
ate_model = AutoModelForTokenClassification.from_pretrained(
    ate_model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(device)

if USE_GRADIENT_CHECKPOINTING and hasattr(ate_model, "gradient_checkpointing_enable"):
    ate_model.gradient_checkpointing_enable()

ate_output_dir = "/content/model_cache_phase2/ate_teacher_phase2/trainer"
ate_checkpoint_drive_dir = "/content/drive/MyDrive/ate_teacher_phase2/checkpoints"
ate_best_dir = "/content/drive/MyDrive/ate_teacher_phase2/best_eval_loss"
ate_final_dir = "/content/drive/MyDrive/ate_teacher_phase2/final"

ate_args = build_training_args(
    output_dir=ate_output_dir,
    lr=3e-6,
    epochs=ATE_EPOCHS,
    train_bs=ATE_TRAIN_BATCH_SIZE,
    eval_bs=ATE_EVAL_BATCH_SIZE,
    grad_accum=ATE_GRAD_ACCUM,
    logging_steps=250,
    eval_steps=250,
    save_steps=250,
    metric_for_best_model="eval_span_f1",
    greater_is_better=True,
    prediction_loss_only=False,
)

ate_trainer = WeightedTokenTrainer(
    class_weights=class_weights,
    model=ate_model,
    args=ate_args,
    train_dataset=ate_train_dataset,
    eval_dataset=ate_val_dataset,
    data_collator=ate_data_collator,

    compute_metrics=compute_ate_span_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ),
        MemoryCleanupCallback(),
        DriveCheckpointSyncCallback(
            drive_dir=ate_checkpoint_drive_dir,
            tokenizer=ate_tokenizer,
            keep_last=DRIVE_CHECKPOINT_KEEP_LAST,
            enabled=SYNC_CHECKPOINTS_TO_DRIVE,
            sync_every_n_checkpoints=DRIVE_SYNC_EVERY_N_CHECKPOINTS,
        ),
    ],
    **trainer_tokenizer_kwargs(ate_tokenizer),
)

ate_resume_checkpoint = prepare_resume_checkpoint(
    ate_checkpoint_drive_dir,
    ate_output_dir,
    enabled=RESUME_FROM_DRIVE_CHECKPOINT,
)

ate_train_result = ate_trainer.train(resume_from_checkpoint=ate_resume_checkpoint)

# Because load_best_model_at_end=True, this saves the best eval-loss model, not merely the final step.
ate_trainer.save_model(ate_best_dir)
ate_tokenizer.save_pretrained(ate_best_dir)

ate_trainer.save_model(ate_final_dir)
ate_tokenizer.save_pretrained(ate_final_dir)

with open(os.path.join(ate_best_dir, "training_summary.json"), "w") as f:
    json.dump(
        {
            "best_metric": float(ate_trainer.state.best_metric) if ate_trainer.state.best_metric is not None else None,
            "best_model_checkpoint": ate_trainer.state.best_model_checkpoint,
            "global_step": int(ate_trainer.state.global_step),
            "max_epochs": ATE_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "checkpoint_steps": 1000,
            "drive_sync_every_n_checkpoints": DRIVE_SYNC_EVERY_N_CHECKPOINTS,
            "train_result": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in ate_train_result.metrics.items()},
        },
        f,
        indent=2,
    )

cleanup_memory()
print("Best ATE eval_span_f1:", ate_trainer.state.best_metric)
print("Best local checkpoint:", ate_trainer.state.best_model_checkpoint)
print("Saved best ATE teacher to Drive:", ate_best_dir)
print("Saved final-loaded ATE teacher to:", ate_final_dir)
print("Drive checkpoints:", ate_checkpoint_drive_dir)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss,Span Precision,Span Recall,Span F1
250,0.501419,0.397086,0.084281,0.502950,0.144369
500,0.472047,0.345740,0.207362,0.498525,0.292894
750,0.373282,0.241187,0.290574,0.604720,0.392532
1000,0.231736,0.182655,0.431881,0.687316,0.530450
1250,0.146263,0.182061,0.480987,0.690265,0.566929


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Synced checkpoint to Drive: /content/drive/MyDrive/ate_teacher_phase2/checkpoints/checkpoint-250


...


Synced checkpoint to Drive: /content/drive/MyDrive/ate_teacher_phase2/checkpoints/checkpoint-2750


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best ATE eval_span_f1: 0.6186440677966102
Best local checkpoint: /content/model_cache_phase2/ate_teacher_phase2/trainer/checkpoint-2250
Saved best ATE teacher to Drive: /content/drive/MyDrive/ate_teacher_phase2/best_eval_loss
Saved final-loaded ATE teacher to: /content/drive/MyDrive/ate_teacher_phase2/final
Drive checkpoints: /content/drive/MyDrive/ate_teacher_phase2/checkpoints


## 14. ATE evaluation on test

In [ ]:
ate_best_dir = "/content/drive/MyDrive/ate_teacher_phase2/best_eval_loss"

ate_tokenizer = AutoTokenizer.from_pretrained(ate_best_dir, use_fast=True)
ate_model = AutoModelForTokenClassification.from_pretrained(ate_best_dir).to(device)
ate_model.eval()

def normalize_span(span):
    span = clean_text(span)
    span = re.sub(r"\s+", " ", span)
    return span.strip(" \t\n\r.,;:!?()[]{}\"'")

def labels_to_spans(tokens, labels, confs=None):
    spans = []
    i = 0

    while i < len(labels):
        if labels[i] == "B-ASP":
            start = i
            i += 1

            while i < len(labels) and labels[i] == "I-ASP":
                i += 1

            end = i - 1
            text = normalize_span(" ".join(tokens[start:end+1]))

            if text:
                if confs is not None:
                    span_conf = float(np.mean(confs[start:end+1]))
                else:
                    span_conf = 1.0

                spans.append({
                    "start": start,
                    "end": end,
                    "text": text,
                    "conf": span_conf,
                })
        else:
            i += 1

    return spans

def reference_span_set(row):
    return {x["text"] for x in labels_to_spans(row["tokens"], row["word_labels"])}

def predict_ate_for_token_batches(token_rows, batch_size=32 if torch.cuda.is_available() else 8):
    predictions = []

    for start in tqdm(range(0, len(token_rows), batch_size), desc="ATE test predicting"):
        batch_tokens = token_rows[start:start + batch_size]
        enc = ate_tokenizer(
            batch_tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=ATE_MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        word_ids_batch = [enc.word_ids(batch_index=i) for i in range(len(batch_tokens))]
        enc_on_device = {k: v.to(device) for k, v in enc.items()}

        with torch.inference_mode():
            logits = ate_model(**enc_on_device).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            pred_ids = probs.argmax(axis=-1)

        for batch_idx, tokens in enumerate(batch_tokens):
            word_ids = word_ids_batch[batch_idx]
            word_labels = ["O"] * len(tokens)
            word_confs = [0.0] * len(tokens)
            seen = set()

            for tok_idx, word_id in enumerate(word_ids):
                if word_id is None or word_id in seen:
                    continue

                seen.add(word_id)

                pred_id = int(pred_ids[batch_idx, tok_idx])
                word_labels[word_id] = id2label[pred_id]
                word_confs[word_id] = float(probs[batch_idx, tok_idx, pred_id])

            predictions.append({
                "tokens": tokens,
                "labels": word_labels,
                "confs": word_confs,
                "spans": labels_to_spans(tokens, word_labels, word_confs),
            })

    return predictions

def predict_ate_for_tokens(tokens):
    return predict_ate_for_token_batches([tokens], batch_size=1)[0]

def evaluate_span_rows(rows):
    tp = fp = fn = 0

    for item in rows:
        reference = item["reference"]
        pred = item["pred"]

        tp += len(reference & pred)
        fp += len(pred - reference)
        fn += len(reference - pred)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

thresholds = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]
threshold_results = []

ate_test_token_rows = ate_test_df["tokens"].tolist()
ate_test_preds = predict_ate_for_token_batches(ate_test_token_rows)

all_test_predictions = [
    {"row": row, "pred": pred}
    for (_, row), pred in zip(ate_test_df.iterrows(), ate_test_preds)
]

for threshold in thresholds:
    rows = []

    for item in all_test_predictions:
        row = item["row"]
        pred = item["pred"]

        pred_set = {s["text"] for s in pred["spans"] if s["conf"] >= threshold}

        rows.append({
            "reference": reference_span_set(row),
            "pred": pred_set,
        })

    m = evaluate_span_rows(rows)
    m["threshold"] = threshold
    threshold_results.append(m)

ate_threshold_df = pd.DataFrame(threshold_results)
display(ate_threshold_df[[
    "threshold", "precision", "recall", "f1", "tp", "fp", "fn"
]])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ATE test predicting:   0%|          | 0/15 [00:00<?, ?it/s]

,threshold,precision,recall,f1,tp,fp,fn
0,0.00,0.594714,0.640823,0.616908,405,276,227
1,0.50,0.596750,0.639241,0.617265,404,273,228
2,0.70,0.637288,0.594937,0.615385,376,214,256
3,0.80,0.665399,0.553797,0.604491,350,176,282
4,0.90,0.751825,0.488924,0.592522,309,102,323
5,0.95,0.792994,0.393987,0.526427,249,65,383


## 15. Sample ATE predictions

In [ ]:

sample_df = ate_test_df.sample(min(12, len(ate_test_df)), random_state=7)

for _, row in sample_df.iterrows():
    pred = predict_ate_for_tokens(row["tokens"])
    pred_spans = [s for s in pred["spans"] if s["conf"] >= 0.5]

    print("Sentence: ", row[sentence_col])
    print("Aspects:  ", row[aspects_col])
    print("Pred:     ", pred_spans)
    print("Labels:   ", list(zip(row["tokens"], pred["labels"], [round(x, 3) for x in pred["confs"]])))
    print("-" * 100)


ATE test predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence:  thoroughly enjoyed this intelligent and quietly atmospheric mystery
Aspects:   ['mystery']
Pred:      [{'start': 7, 'end': 7, 'text': 'mystery', 'conf': 0.8987049460411072}]
Labels:    [('thoroughly', 'O', 0.994), ('enjoyed', 'O', 0.988), ('this', 'O', 0.993), ('intelligent', 'O', 0.989), ('and', 'O', 0.995), ('quietly', 'O', 0.994), ('atmospheric', 'O', 0.987), ('mystery', 'B-ASP', 0.899)]
----------------------------------------------------------------------------------------------------


ATE test predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence:  good quality coax good quality coax
Aspects:   ['coax']
Pred:      [{'start': 1, 'end': 1, 'text': 'quality', 'conf': 0.8901364803314209}, {'start': 2, 'end': 2, 'text': 'coax', 'conf': 0.9919740557670593}, {'start': 4, 'end': 4, 'text': 'quality', 'conf': 0.8701821565628052}, {'start': 5, 'end': 5, 'text': 'coax', 'conf': 0.9876147508621216}]
Labels:    [('good', 'O', 0.965), ('quality', 'B-ASP', 0.89), ('coax', 'B-ASP', 0.992), ('good', 'O', 0.972), ('quality', 'B-ASP', 0.87), ('coax', 'B-ASP', 0.988)]
----------------------------------------------------------------------------------------------------


ATE test predicting:   0%|          | 0/1 [00:00<?, ?it/s]

...

Sentence:  can find almost [GENERIC_NOUN] great app
Aspects:   ['app']
Pred:      [{'start': 5, 'end': 5, 'text': 'app', 'conf': 0.9926726222038269}]
Labels:    [('can', 'O', 0.997), ('find', 'O', 0.994), ('almost', 'O', 0.995), ('GENERIC_NOUN', 'O', 0.989), ('great', 'O', 0.99), ('app', 'B-ASP', 0.993)]
----------------------------------------------------------------------------------------------------


ATE test predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence:  also the app claims that you can watch the shows anywhere but when you try to watch without wifi it just kicks you off the app
Aspects:   ['app']
Pred:      [{'start': 2, 'end': 2, 'text': 'app', 'conf': 0.9824150204658508}, {'start': 25, 'end': 25, 'text': 'app', 'conf': 0.9369533658027649}]
Labels:    [('also', 'O', 0.996), ('the', 'O', 0.993), ('app', 'B-ASP', 0.982), ('claims', 'O', 0.994), ('that', 'O', 0.997), ('you', 'O', 0.997), ('can', 'O', 0.997), ('watch', 'O', 0.966), ('the', 'O', 0.997), ('shows', 'O', 0.599), ('anywhere', 'O', 0.997), ('but', 'O', 0.998), ('when', 'O', 0.997), ('you', 'O', 0.998), ('try', 'O', 0.997), ('to', 'O', 0.997), ('watch', 'O', 0.938), ('without', 'O', 0.997), ('wifi', 'O', 0.706), ('it', 'O', 0.996), ('just', 'O', 0.997), ('kicks', 'O', 0.997), ('you', 'O', 0.998), ('off', 'O', 0.997), ('the', 'O', 0.996), ('app', 'B-ASP', 0.937)]
----------------------------------------------------------------------------------------------------


ATE test predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence:  keeps you engaged and has intricate plot
Aspects:   ['plot']
Pred:      [{'start': 6, 'end': 6, 'text': 'plot', 'conf': 0.9652782678604126}]
Labels:    [('keeps', 'O', 0.992), ('you', 'O', 0.993), ('engaged', 'O', 0.982), ('and', 'O', 0.991), ('has', 'O', 0.994), ('intricate', 'O', 0.992), ('plot', 'B-ASP', 0.965)]
----------------------------------------------------------------------------------------------------
